# Tennis Match Inference Pipeline
**Detection (Roboflow)** + **Action Classification (YOLOv11-cls)** → Annotated Video Output

In [ ]:
# 1. Kurulum
!pip install ultralytics roboflow supervision -q

In [ ]:
# 2. Import'lar ve setup
import cv2
import numpy as np
import supervision as sv
from ultralytics import YOLO
from roboflow import Roboflow
from kaggle_secrets import UserSecretsClient
from pathlib import Path
from collections import defaultdict, deque
import time

# Paths
OUTPUT_DIR = Path('/kaggle/working/output')
OUTPUT_DIR.mkdir(exist_ok=True)

print('Setup tamamlandı.')

In [ ]:
# 3. Roboflow'dan detection modelini indir
secrets = UserSecretsClient()
api_key = secrets.get_secret('ROBOFLOW_API_KEY')

rf = Roboflow(api_key=api_key)

# Roboflow projen: workspace ve proje adını kendi bilgilerinle değiştir
project = rf.workspace('pelins-workspace-a7gej').project('tennis-tracker-duufq-dntba')
version = project.version(1)  # hangi version'ı train ettiysen onu yaz
model_path = version.download('yolov11')

print(f'Detection modeli indirildi: {model_path}')

In [ ]:
import os

print('working içeriği:')
for item in os.listdir('/kaggle/working'):
    print(item)

In [ ]:
from ultralytics import YOLO
import os

DATASET_YAML = '/kaggle/working/tennis-tracker-1/data.yaml'

detection_model = YOLO('yolo11n.pt')
results = detection_model.train(
    data=DATASET_YAML,
    epochs=50,
    imgsz=640,
    batch=8,
    patience=10,
    save=True,
    save_period=5,
    project='/kaggle/working/runs',
    name='tennis_detection',
    exist_ok=True,
    verbose=False
)

DETECT_MODEL_PATH = '/kaggle/working/runs/tennis_detection/weights/best.pt'
detection_model = YOLO(DETECT_MODEL_PATH)
print(f'Classes: {detection_model.names}')

In [ ]:
# Modelleri kalıcı olarak kaydet
import shutil
import os

SAVE_DIR = '/kaggle/working/output/models'
os.makedirs(SAVE_DIR, exist_ok=True)

# Detection modeli
shutil.copy(
    '/kaggle/working/runs/tennis_detection/weights/best.pt',
    f'{SAVE_DIR}/tennis_detection_best.pt'
)

# Classifier zaten input'ta, onu da kopyala
shutil.copy(
    '/kaggle/input/models/ecepelin/tennis-action-classifier-best-pt/pytorch/default/1/tennis_action_classifier_best.pt',
    f'{SAVE_DIR}/tennis_action_classifier_best.pt'
)

print('Kaydedilen dosyalar:')
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(f'{SAVE_DIR}/{f}') / (1024*1024)
    print(f'  {f}: {size:.1f} MB')

In [ ]:
# Cell 4 - Modelleri yükle
detection_model = YOLO(DETECT_MODEL_PATH)
print(f'Classes: {detection_model.names}')

CLS_MODEL_PATH = '/kaggle/input/models/ecepelin/tennis-action-classifier-best-pt/pytorch/default/1/tennis_action_classifier_best.pt'
cls_model = YOLO(CLS_MODEL_PATH)
print(f'Action classes: {cls_model.names}')

In [ ]:
# Cell 5 - Dataset görüntülerinden test videosu oluştur
import cv2
import glob
import random

# Mevcut train görüntülerini kullan
image_paths = glob.glob('/kaggle/working/tennis-tracker-1/train/images/*.jpg')
image_paths = sorted(image_paths)[:150]  # İlk 150 kare yeterli

print(f'{len(image_paths)} görüntü bulundu')

# İlk görüntüden boyutu al
sample = cv2.imread(image_paths[0])
h, w = sample.shape[:2]

# Video oluştur
VIDEO_PATH = '/kaggle/working/test_match.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(VIDEO_PATH, fourcc, 15, (w, h))  # 15 FPS

for img_path in image_paths:
    frame = cv2.imread(img_path)
    if frame is not None:
        out.write(frame)

out.release()

size_mb = os.path.getsize(VIDEO_PATH) / (1024*1024)
print(f'Test videosu oluşturuldu: {VIDEO_PATH} ({size_mb:.1f} MB)')
print(f'Süre: ~{len(image_paths)/15:.1f} saniye')

In [ ]:
# 6. Video bilgilerini kontrol et
cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = total_frames / fps

print(f'FPS: {fps}')
print(f'Resolution: {width}x{height}')
print(f'Total frames: {total_frames}')
print(f'Duration: {duration:.1f} seconds')

cap.release()

In [ ]:
# 7. Analytics tracker class
class TennisAnalytics:
    def __init__(self):
        # Shot sayaçları (player id bazında)
        self.shot_counts = defaultdict(lambda: defaultdict(int))
        # Son N frame'deki action'lar (smoothing için)
        self.action_history = defaultdict(lambda: deque(maxlen=10))
        # Ball pozisyon geçmişi
        self.ball_positions = deque(maxlen=30)
        # Rally sayacı
        self.rally_count = 0
        self.last_ball_seen = 0
        self.frame_count = 0
        
    def update_ball(self, ball_center, frame_idx):
        if ball_center is not None:
            self.ball_positions.append((frame_idx, ball_center))
            self.last_ball_seen = frame_idx
    
    def update_action(self, player_id, action):
        self.action_history[player_id].append(action)
        self.shot_counts[player_id][action] += 1
    
    def get_smoothed_action(self, player_id):
        """Son 10 frame'deki en sık görülen action'ı döndür (temporal smoothing)"""
        history = list(self.action_history[player_id])
        if not history:
            return None
        return max(set(history), key=history.count)
    
    def get_summary(self):
        summary = {}
        for player_id, shots in self.shot_counts.items():
            total = sum(shots.values())
            summary[player_id] = {
                'total_shots': total,
                'shot_distribution': {k: f'{v/total*100:.1f}%' for k, v in shots.items()}
            }
        return summary

analytics = TennisAnalytics()
print('Analytics tracker hazır.')

In [ ]:
# 8. Annotation helper fonksiyonları

# Renkler (BGR)
COLORS = {
    'player-front': (0, 255, 0),      # Yeşil
    'player-back':  (255, 165, 0),    # Turuncu
    'tennis-ball':  (0, 255, 255),    # Sarı
    'person':       (128, 128, 128),  # Gri
}

ACTION_COLORS = {
    'forehand':       (0, 200, 0),
    'backhand':       (0, 0, 255),
    'serve':          (255, 0, 255),
    'ready_position': (255, 200, 0),
}

def draw_detection(frame, bbox, class_name, conf, action=None, player_id=None):
    x1, y1, x2, y2 = map(int, bbox)
    color = COLORS.get(class_name, (255, 255, 255))
    
    # Bounding box
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
    
    # Label
    label = f'{class_name} {conf:.2f}'
    if player_id is not None:
        label = f'P{player_id} {label}'
    
    label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)[0]
    cv2.rectangle(frame, (x1, y1 - label_size[1] - 4), 
                  (x1 + label_size[0], y1), color, -1)
    cv2.putText(frame, label, (x1, y1 - 2),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
    
    # Action overlay (player üzerinde)
    if action and 'player' in class_name:
        action_color = ACTION_COLORS.get(action, (255, 255, 255))
        action_label = action.upper().replace('_', ' ')
        cv2.putText(frame, action_label, (x1, y2 + 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, action_color, 2)
    
    return frame

def draw_scoreboard(frame, analytics, frame_idx, fps):
    """Sol üst köşeye istatistik paneli çiz"""
    h, w = frame.shape[:2]
    panel_w, panel_h = 280, 150
    
    # Yarı şeffaf arka plan
    overlay = frame.copy()
    cv2.rectangle(overlay, (10, 10), (panel_w, panel_h), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
    
    # Zaman
    seconds = frame_idx / fps
    cv2.putText(frame, f'Time: {seconds:.1f}s', (15, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    
    # Her player için shot sayıları
    y_offset = 50
    for player_id, shots in analytics.shot_counts.items():
        total = sum(shots.values())
        cv2.putText(frame, f'P{player_id}: {total} shots', (15, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1)
        y_offset += 18
        
        # En sık kullanılan shot
        if shots:
            top_shot = max(shots, key=shots.get)
            color = ACTION_COLORS.get(top_shot, (255, 255, 255))
            cv2.putText(frame, f'  Dominant: {top_shot}', (15, y_offset),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
            y_offset += 18
    
    return frame

def draw_ball_trail(frame, ball_positions):
    """Top yolunu çiz"""
    positions = list(ball_positions)
    for i in range(1, len(positions)):
        alpha = i / len(positions)  # Soldan sağa opaklık artar
        color = (0, int(255 * alpha), int(255 * alpha))
        thickness = max(1, int(3 * alpha))
        cv2.line(frame, positions[i-1][1], positions[i][1], color, thickness)
    return frame

print('Annotation fonksiyonları hazır.')

In [ ]:
# Modelleri yeniden yükle
DETECT_MODEL_PATH = '/kaggle/working/runs/tennis_detection/weights/best.pt'
detection_model = YOLO(DETECT_MODEL_PATH)

CLS_MODEL_PATH = '/kaggle/input/models/ecepelin/tennis-action-classifier-best-pt/pytorch/default/1/tennis_action_classifier_best.pt'
cls_model = YOLO(CLS_MODEL_PATH)

print(f'Detection: {detection_model.names}')
print(f'Classifier: {cls_model.names}')

In [ ]:
# 9. Ana inference pipeline

CONF_THRESHOLD = 0.4      # Detection confidence
CLS_CONF_THRESHOLD = 0.6  # Classification confidence
PROCESS_EVERY_N = 2       # Her N frame'de bir işle (hız için)
OUTPUT_VIDEO = str(OUTPUT_DIR / 'tennis_annotated.mp4')

# Video reader/writer
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

analytics = TennisAnalytics()
frame_idx = 0
player_actions = {}  # player_id → smoothed action

print(f'Pipeline başladı: {VIDEO_PATH}')
print(f'Output: {OUTPUT_VIDEO}')
start_time = time.time()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # Her N frame'de bir detection yap (performance)
    if frame_idx % PROCESS_EVERY_N == 0:
        
        # --- DETECTION ---
        det_results = detection_model(
            frame, 
            conf=CONF_THRESHOLD, 
            verbose=False
        )[0]
        
        player_id_counter = 0
        
        for i, box in enumerate(det_results.boxes):
            cls_idx = int(box.cls[0])
            class_name = detection_model.names[cls_idx]
            conf = float(box.conf[0])
            bbox = box.xyxy[0].cpu().numpy()
            
            # person class'ını atla
            if class_name == 'person':
                continue
            
            # Ball tracking
            if class_name == 'tennis-ball':
                x1, y1, x2, y2 = map(int, bbox)
                center = ((x1 + x2) // 2, (y1 + y2) // 2)
                analytics.update_ball(center, frame_idx)
                frame = draw_detection(frame, bbox, class_name, conf)
                continue
            
            # Player detection → action classification
            if 'player' in class_name:
                player_id = player_id_counter
                player_id_counter += 1
                
                # Player crop'u al
                x1, y1, x2, y2 = map(int, bbox)
                # Sınır kontrolü
                x1 = max(0, x1)
                y1 = max(0, y1)
                x2 = min(width, x2)
                y2 = min(height, y2)
                
                player_crop = frame[y1:y2, x1:x2]
                
                action = None
                if player_crop.size > 0 and player_crop.shape[0] > 20 and player_crop.shape[1] > 20:
                    # --- ACTION CLASSIFICATION ---
                    cls_result = cls_model(
                        player_crop, 
                        imgsz=224, 
                        verbose=False
                    )[0]
                    
                    top1_conf = float(cls_result.probs.top1conf)
                    
                    if top1_conf >= CLS_CONF_THRESHOLD:
                        action = cls_model.names[cls_result.probs.top1]
                        analytics.update_action(player_id, action)
                
                # Temporal smoothing
                smoothed_action = analytics.get_smoothed_action(player_id)
                player_actions[player_id] = smoothed_action
                
                frame = draw_detection(frame, bbox, class_name, conf, 
                                       smoothed_action, player_id)
    
    # Ball trail çiz
    frame = draw_ball_trail(frame, analytics.ball_positions)
    
    # Scoreboard çiz
    frame = draw_scoreboard(frame, analytics, frame_idx, fps)
    
    out.write(frame)
    frame_idx += 1
    
    # Progress
    if frame_idx % 100 == 0:
        elapsed = time.time() - start_time
        print(f'Frame {frame_idx} işlendi | Süre: {elapsed:.1f}s')

cap.release()
out.release()

total_time = time.time() - start_time
print(f'\nPipeline tamamlandı!')
print(f'Toplam süre: {total_time:.1f}s')
print(f'Output: {OUTPUT_VIDEO}')

In [ ]:
# 10. Analytics özeti
print('=== MATCH ANALYTICS ===')
summary = analytics.get_summary()

for player_id, data in summary.items():
    print(f'\nPlayer {player_id}:')
    print(f'  Total shots detected: {data["total_shots"]}')
    print(f'  Shot distribution:')
    for shot, pct in data['shot_distribution'].items():
        print(f'    {shot}: {pct}')

print(f'\nTotal ball positions tracked: {len(analytics.ball_positions)}')

In [ ]:
# 11. Output video'yu kontrol et
import os

output_size = os.path.getsize(OUTPUT_VIDEO) / (1024 * 1024)
print(f'Output video: {OUTPUT_VIDEO}')
print(f'Boyut: {output_size:.1f} MB')
print('Kaggle Output sekmesinden indirebilirsin.')

# İlk frame'i görsel olarak kontrol et
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

cap_check = cv2.VideoCapture(OUTPUT_VIDEO)
ret, sample_frame = cap_check.read()
cap_check.release()

if ret:
    sample_frame_rgb = cv2.cvtColor(sample_frame, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(14, 8))
    plt.imshow(sample_frame_rgb)
    plt.title('Output Video - İlk Frame')
    plt.axis('off')
    plt.show()

In [ ]:
print(dir(sv))

In [ ]:
import supervision as sv
from collections import defaultdict, deque
import numpy as np

tracker = sv.ByteTrack(
    track_activation_threshold=0.3,
    lost_track_buffer=30,
    minimum_matching_threshold=0.8,
    frame_rate=15
)

class MatchState:
    def __init__(self):
        self.scores = {0: 0, 1: 0}
        self.rally_active = False
        self.rally_length = 0
        self.last_shot_player = None
        self.ball_missing_frames = 0
        self.ball_missing_threshold = 4
        self.shot_history = []
        self.point_history = []
        self.player_track_actions = defaultdict(lambda: deque(maxlen=15))
        self.track_to_player = {}
        self.player_slot_counter = 0
        self.last_point_frame = -50
        self.point_cooldown = 30

    def get_player_slot(self, track_id):
        if track_id not in self.track_to_player:
            if self.player_slot_counter < 2:
                self.track_to_player[track_id] = self.player_slot_counter
                self.player_slot_counter += 1
            else:
                return None
        return self.track_to_player[track_id]

    def update(self, frame_idx, ball_detected, player_shots):
        if ball_detected:
            self.ball_missing_frames = 0
            if not self.rally_active and len(player_shots) > 0:
                self.rally_active = True
                self.rally_length = 0
        else:
            self.ball_missing_frames += 1

        for track_id, action in player_shots:
            slot = self.get_player_slot(track_id)
            if slot is None:
                continue
            if action not in ['ready_position']:
                self.player_track_actions[slot].append(action)
                self.shot_history.append((frame_idx, slot, action))
                self.last_shot_player = slot
                self.rally_length += 1

        winner = None
        reason = None

        if frame_idx - self.last_point_frame < self.point_cooldown:
            return None

        if self.rally_active and self.ball_missing_frames >= self.ball_missing_threshold:
            if self.last_shot_player is not None:
                loser = self.last_shot_player
                winner = 1 - loser
                reason = 'ball_out'
            self.rally_active = False
            self.rally_length = 0
            self.last_shot_player = None
        elif self.rally_active:
            for track_id, action in player_shots:
                slot = self.get_player_slot(track_id)
                if slot is None:
                    continue
                history = list(self.player_track_actions[slot])
                if history.count('serve') >= 3 and len(history) >= 5:
                    winner = 1 - slot
                    reason = 'double_fault'
                    self.player_track_actions[slot].clear()
                    break

        if winner is not None:
            self.scores[winner] = self.scores.get(winner, 0) + 1
            self.last_point_frame = frame_idx
            self.point_history.append((frame_idx, winner, reason))
            print(f'Frame {frame_idx}: Player {winner} sayı aldı ({reason}) | Skor: {dict(self.scores)}')

        return winner

    def get_score_display(self):
        return f"P0: {self.scores.get(0,0)} | P1: {self.scores.get(1,0)}"

match_state = MatchState()
print('Tracking + Match state hazır.')

In [ ]:
# Tracking entegreli inference pipeline
import cv2
import time
import os

CONF_THRESHOLD = 0.4
CLS_CONF_THRESHOLD = 0.5
OUTPUT_VIDEO = '/kaggle/working/output/tennis_tracked.mp4'

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

# State sıfırla
tracker = sv.ByteTrack(track_activation_threshold=0.3, lost_track_buffer=30,
                        minimum_matching_threshold=0.8, frame_rate=int(fps))
match_state = MatchState()
match_state.scores = {}
ball_positions = deque(maxlen=30)
frame_idx = 0
start_time = time.time()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # --- DETECTION ---
    det_results = detection_model(frame, conf=CONF_THRESHOLD, verbose=False)[0]

    # Supervision Detections formatına çevir
    boxes = det_results.boxes.xyxy.cpu().numpy()
    confs = det_results.boxes.conf.cpu().numpy()
    class_ids = det_results.boxes.cls.cpu().numpy().astype(int)

    detections = sv.Detections(
        xyxy=boxes,
        confidence=confs,
        class_id=class_ids
    )

    # Ball ve player'ları ayır
    ball_mask = class_ids == 2  # tennis-ball
    player_mask = class_ids != 2

    ball_detections = detections[ball_mask]
    player_detections = detections[player_mask]

    # --- TRACKING (sadece player'lar için) ---
    if len(player_detections) > 0:
        tracked = tracker.update_with_detections(player_detections)
    else:
        tracked = sv.Detections.empty()

    # Ball tracking
    ball_detected = len(ball_detections) > 0
    if ball_detected:
        bx1, by1, bx2, by2 = map(int, ball_detections.xyxy[0])
        ball_center = ((bx1 + bx2) // 2, (by1 + by2) // 2)
        ball_positions.append(ball_center)
        cv2.circle(frame, ball_center, 6, (0, 255, 255), -1)

        # Ball trail
        positions = list(ball_positions)
        for i in range(1, len(positions)):
            alpha = i / len(positions)
            color = (0, int(255 * alpha), int(255 * alpha))
            cv2.line(frame, positions[i-1], positions[i], color, 2)

    # --- ACTION CLASSIFICATION + DRAW ---
    player_shots = []

    for i in range(len(tracked)):
        x1, y1, x2, y2 = map(int, tracked.xyxy[i])
        track_id = int(tracked.tracker_id[i]) if tracked.tracker_id is not None else i
        cls_idx = int(tracked.class_id[i])
        class_name = detection_model.names[cls_idx]
        conf = float(tracked.confidence[i])

        # Sınır kontrolü
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(width, x2), min(height, y2)
        player_crop = frame[y1:y2, x1:x2]

        action = None
        if player_crop.size > 0 and player_crop.shape[0] > 20 and player_crop.shape[1] > 20:
            cls_result = cls_model(player_crop, imgsz=224, verbose=False)[0]
            top1_conf = float(cls_result.probs.top1conf)
            if top1_conf >= CLS_CONF_THRESHOLD:
                action = cls_model.names[cls_result.probs.top1]
                player_shots.append((track_id, action))

        # Renk: player-front yeşil, player-back turuncu
        color = (0, 255, 0) if cls_idx == 1 else (255, 165, 0)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

        label = f'P{track_id} {class_name} {conf:.2f}'
        cv2.putText(frame, label, (x1, y1 - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)

        if action:
            action_color = {'forehand': (0,200,0), 'backhand': (0,0,255),
                           'serve': (255,0,255), 'ready_position': (200,200,0)}.get(action, (255,255,255))
            cv2.putText(frame, action.upper(), (x1, y2 + 18),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, action_color, 2)



    match_state.update(frame_idx, ball_detected, player_shots)

    # --- SCOREBOARD ---
    overlay = frame.copy()
    cv2.rectangle(overlay, (8, 8), (300, 100), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, frame)

    cv2.putText(frame, 'SCORE', (15, 28),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 0), 1)

    y = 50
    for pid, score in match_state.scores.items():
        cv2.putText(frame, match_state.get_score_display(), (15, 50),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)

    rally_status = 'RALLY' if match_state.rally_active else 'WAITING'
    rally_color = (0, 255, 0) if match_state.rally_active else (100, 100, 100)
    cv2.putText(frame, rally_status, (width - 120, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, rally_color, 2)

    out.write(frame)
    frame_idx += 1

cap.release()
out.release()

print(f'Pipeline tamamlandı! Süre: {time.time()-start_time:.1f}s')
print(f'Output: {OUTPUT_VIDEO}')
print(f'\nFinal Skor: {dict(match_state.scores)}')
print(f'Toplam point: {len(match_state.point_history)}')
for frame_n, winner, reason in match_state.point_history:
    print(f'  Frame {frame_n}: Player {winner} kazandı ({reason})')

Testing with real match videos

In [ ]:
import os

BASE = '/kaggle/input/datasets/gastonarielfrancois/tenis-backview'

for root, dirs, files in os.walk(BASE):
    level = root.replace(BASE, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files[:5]:
        print(f'{indent}  {f}')
    if level > 2:
        break

In [ ]:
import glob
import cv2
import os

# Görüntüleri maç adına göre grupla ve sırala
image_paths = glob.glob('/kaggle/working/tennis-tracker-1/train/images/*.jpg')

# Dosya adlarına bak
for p in sorted(image_paths)[:10]:
    print(os.path.basename(p))

In [ ]:
import glob
import re
import cv2
import os

image_paths = glob.glob('/kaggle/working/tennis-tracker-1/train/images/*.jpg')

# Sadece Basel maçını al, bir augmentation versiyonu, saniyeye göre sırala
basel_images = {}
for p in image_paths:
    name = os.path.basename(p)
    if 'Basel-2019-Tuesday' in name:
        # Saniye numarasını çıkar
        match = re.search(r'Highlights-(\d+)-', name)
        if match:
            second = int(match.group(1))
            # Her saniyeden sadece bir tane al
            if second not in basel_images:
                basel_images[second] = p

# Saniyeye göre sırala
sorted_paths = [basel_images[s] for s in sorted(basel_images.keys())]
print(f'Toplam ardışık frame: {len(sorted_paths)}')
for p in sorted_paths[:10]:
    print(os.path.basename(p))

In [ ]:
import glob
import re
import cv2
import os

image_paths = glob.glob('/kaggle/working/tennis-tracker-1/train/images/*.jpg')

# Tüm maçları grupla
match_images = {}
for p in image_paths:
    name = os.path.basename(p)
    # Maç adını çıkar (saniye ve augmentation kısmı hariç)
    parts = name.split('-')
    # Son iki segment: saniye ve augmentation → onları çıkar
    match_name = '-'.join(parts[:-2])
    second_match = re.search(r'-(\d+)-\d+_jpg', name)
    if second_match:
        second = int(second_match.group(1))
        if match_name not in match_images:
            match_images[match_name] = {}
        if second not in match_images[match_name]:
            match_images[match_name][second] = p

# Her maçı yazdır
print('Maç başına frame sayısı:')
for match_name, frames in sorted(match_images.items()):
    print(f'  {match_name}: {len(frames)} frame')

# En fazla frame'e sahip maçı seç
best_match = max(match_images, key=lambda m: len(match_images[m]))
sorted_paths = [match_images[best_match][s] for s in sorted(match_images[best_match].keys())]
print(f'\nSeçilen maç: {best_match}')
print(f'Frame sayısı: {len(sorted_paths)}')

In [ ]:
# Tüm maçlardan tüm frame'leri topla, maç+saniye sırasına göre sırala
all_sorted = []
for match_name, frames in sorted(match_images.items()):
    for second in sorted(frames.keys()):
        all_sorted.append(frames[second])

print(f'Toplam frame: {len(all_sorted)}')

# Video oluştur
sample = cv2.imread(all_sorted[0])
h, w = sample.shape[:2]

VIDEO_PATH = '/kaggle/working/test_match.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_vid = cv2.VideoWriter(VIDEO_PATH, fourcc, 2, (w, h))  # 2 FPS — görüntüler arası 1 saniye var

for img_path in all_sorted:
    frame = cv2.imread(img_path)
    if frame is not None:
        out_vid.write(frame)

out_vid.release()
print(f'Video oluşturuldu: {len(all_sorted)} frame, {len(all_sorted)/2:.0f} saniye')
print(f'Path: {VIDEO_PATH}')

In [ ]:
# Tracking entegreli inference pipeline
import cv2
import time
import os

CONF_THRESHOLD = 0.4
CLS_CONF_THRESHOLD = 0.5
OUTPUT_VIDEO = '/kaggle/working/output/test_match.mp4'

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

# State sıfırla
tracker = sv.ByteTrack(track_activation_threshold=0.3, lost_track_buffer=30,
                        minimum_matching_threshold=0.8, frame_rate=2)  # 2 FPS
match_state = MatchState()
match_state.scores = {}
ball_positions = deque(maxlen=30)
frame_idx = 0
start_time = time.time()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # --- DETECTION ---
    det_results = detection_model(frame, conf=CONF_THRESHOLD, verbose=False)[0]

    # Supervision Detections formatına çevir
    boxes = det_results.boxes.xyxy.cpu().numpy()
    confs = det_results.boxes.conf.cpu().numpy()
    class_ids = det_results.boxes.cls.cpu().numpy().astype(int)

    detections = sv.Detections(
        xyxy=boxes,
        confidence=confs,
        class_id=class_ids
    )

    # Ball ve player'ları ayır
    ball_mask = class_ids == 2  # tennis-ball
    player_mask = class_ids != 2

    ball_detections = detections[ball_mask]
    player_detections = detections[player_mask]

    # --- TRACKING (sadece player'lar için) ---
    if len(player_detections) > 0:
        tracked = tracker.update_with_detections(player_detections)
    else:
        tracked = sv.Detections.empty()

    # Ball tracking
    ball_detected = len(ball_detections) > 0
    if ball_detected:
        bx1, by1, bx2, by2 = map(int, ball_detections.xyxy[0])
        ball_center = ((bx1 + bx2) // 2, (by1 + by2) // 2)
        ball_positions.append(ball_center)
        cv2.circle(frame, ball_center, 6, (0, 255, 255), -1)

        # Ball trail
        positions = list(ball_positions)
        for i in range(1, len(positions)):
            alpha = i / len(positions)
            color = (0, int(255 * alpha), int(255 * alpha))
            cv2.line(frame, positions[i-1], positions[i], color, 2)

    # --- ACTION CLASSIFICATION + DRAW ---
    player_shots = []

    for i in range(len(tracked)):
        x1, y1, x2, y2 = map(int, tracked.xyxy[i])
        track_id = int(tracked.tracker_id[i]) if tracked.tracker_id is not None else i
        cls_idx = int(tracked.class_id[i])
        class_name = detection_model.names[cls_idx]
        conf = float(tracked.confidence[i])

        # Sınır kontrolü
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(width, x2), min(height, y2)
        player_crop = frame[y1:y2, x1:x2]

        action = None
        if player_crop.size > 0 and player_crop.shape[0] > 20 and player_crop.shape[1] > 20:
            cls_result = cls_model(player_crop, imgsz=224, verbose=False)[0]
            top1_conf = float(cls_result.probs.top1conf)
            if top1_conf >= CLS_CONF_THRESHOLD:
                action = cls_model.names[cls_result.probs.top1]
                player_shots.append((track_id, action))

        # Renk: player-front yeşil, player-back turuncu
        color = (0, 255, 0) if cls_idx == 1 else (255, 165, 0)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

        label = f'P{track_id} {class_name} {conf:.2f}'
        cv2.putText(frame, label, (x1, y1 - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)

        if action:
            action_color = {'forehand': (0,200,0), 'backhand': (0,0,255),
                           'serve': (255,0,255), 'ready_position': (200,200,0)}.get(action, (255,255,255))
            cv2.putText(frame, action.upper(), (x1, y2 + 18),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, action_color, 2)



    match_state.update(frame_idx, ball_detected, player_shots)

    # --- SCOREBOARD ---
    overlay = frame.copy()
    cv2.rectangle(overlay, (8, 8), (300, 100), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, frame)

    cv2.putText(frame, 'SCORE', (15, 28),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 0), 1)

    y = 50
    for pid, score in match_state.scores.items():
        cv2.putText(frame, match_state.get_score_display(), (15, 50),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)

    rally_status = 'RALLY' if match_state.rally_active else 'WAITING'
    rally_color = (0, 255, 0) if match_state.rally_active else (100, 100, 100)
    cv2.putText(frame, rally_status, (width - 120, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, rally_color, 2)

    out.write(frame)
    frame_idx += 1

cap.release()
out.release()

print(f'Pipeline tamamlandı! Süre: {time.time()-start_time:.1f}s')
print(f'Output: {OUTPUT_VIDEO}')
print(f'\nFinal Skor: {dict(match_state.scores)}')
print(f'Toplam point: {len(match_state.point_history)}')
for frame_n, winner, reason in match_state.point_history:
    print(f'  Frame {frame_n}: Player {winner} kazandı ({reason})')